# PoC 1 — Better Encoder (A) + Query Expansion (B) + Client-side RRF (D)

**Goal:** Replace the weak all-MiniLM-L6-v2 encoder with `all-mpnet-base-v2`
(768-dim, 2× better retrieval quality), add LLM/rule-based query expansion,
and fuse BM25 + kNN results with Reciprocal Rank Fusion — all 100% free,
no Elasticsearch subscription required.

| Component | Implementation | License |
|-----------|---------------|----------|
| **A – Better encoder** | `all-mpnet-base-v2` (HuggingFace) | Apache-2.0 |
| **B – Query expansion** | Rule-based synonym map + Ollama (optional) | Free |
| **D – RRF fusion** | Pure Python, no ES license needed | — |

**Prerequisite:** Notebook 01 has been run → `neuroimaging` index exists on
`http://localhost:9200`.

In [1]:
import os

# ── Configuration — edit these before running ────────────────────────────────
ES_HOST = os.environ.get("ES_HOST", "http://localhost:9200")
BASE_INDEX = "neuroimaging"          # source index (created by Notebook 01)
POC1_INDEX = "neuroimaging-poc1"     # new index created by this notebook

# swap for "BAAI/bge-large-en-v1.5" (1024d)
ENCODER_MODEL = "all-mpnet-base-v2"
EMBEDDING_DIMS = 768

USE_RRF = True   # True → client-side RRF; False → simple weighted hybrid
RRF_K = 60     # smoothing constant (original paper default)
TOP_K = 5      # results per search

RESULTS_PATH = "poc1_results.json"     # saved for the benchmark in Notebook 07

In [2]:
import json
import time
from tqdm import tqdm
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch, helpers
import warnings
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)


client = Elasticsearch(ES_HOST, request_timeout=120)
assert client.ping(), f"Cannot reach Elasticsearch at {ES_HOST}"
print(f"Connected to ES {client.info()['version']['number']}")

print(f"Loading encoder: {ENCODER_MODEL} ...")
model = SentenceTransformer(ENCODER_MODEL, device='cpu')
actual_dims = model.get_sentence_embedding_dimension()
assert actual_dims == EMBEDDING_DIMS, f"Dim mismatch: expected {EMBEDDING_DIMS}, got {actual_dims}"
print(f"Encoder ready. dims={EMBEDDING_DIMS}")


def show_hits(hits: list[dict], fields=None) -> pd.DataFrame:
    rows = []
    for hit in hits:
        row = {"_score": round(float(hit.get("_score") or 0), 5)}
        src = hit.get("_source", {})
        for f in (fields or ["dataset", "suffix", "Manufacturer",
                             "MagneticFieldStrength", "TaskName", "description_text"]):
            row[f] = src.get(f)
        rows.append(row)
    return pd.DataFrame(rows)

Connected to ES 9.3.0
Loading encoder: all-mpnet-base-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder ready. dims=768


## Option B — Query Expansion

Before sending anything to ES, we expand the query to surface implied domain
vocabulary. Both the BM25 `match` query and the kNN vector benefit simultaneously.

Two backends (tried in order):
1. **Ollama** — fully local LLM (`ollama pull llama3`)
2. **Rule-based** — deterministic synonym map (always available, zero deps)

In [ ]:
import os
import urllib.request as _ureq
import json as _json
import re

_OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_URL = _OLLAMA_HOST + "/api/generate"

OLLAMA_PROMPT = (
    "You are a neuroimaging search assistant. Expand a short user query into richer "
    "BIDS-aware search terms that find the most relevant files in an Elasticsearch "
    "index of neuroimaging metadata.\n\n"
    "BIDS suffixes: bold=BOLD fMRI, T1w=structural anatomical, T2w=T2-weighted, "
    "dwi=Diffusion Weighted Imaging, phasediff/magnitude1/magnitude2/epi=fieldmap, "
    "UNIT1=MP2RAGE uniform T1, T1map=quantitative T1 relaxometry, "
    "asl=Arterial Spin Labeling perfusion, m0scan=ASL M0 reference, pet=PET.\n\n"
    "Key index fields: suffix, task/TaskName, MagneticFieldStrength (Tesla: 1.5/3/7), "
    "Manufacturer/ManufacturersModelName (Siemens/Philips/GE), "
    "RepetitionTime (seconds), EchoTime (ms), InversionTime (ms), FlipAngle (degrees), "
    "PhaseEncodingDirection (AP/PA/RL/LR), InstitutionName.\n\n"
    "Corpus datasets: ds001–ds116 (fMRI/structural), 7t_trt (7T reliability), "
    "qmri_mp2rage/qmri_irt1/qmri_mese/qmri_vfa/qmri_mtsat (quantitative MRI), "
    "asl001–asl005 (perfusion), ds000117 (fMRI+MEG), eeg_rest_fmri (EEG-fMRI).\n\n"
    "Rules: expand abbreviations (BOLD→Blood Oxygen Level Dependent, DWI→diffusion "
    "tensor imaging); add field=value tokens (3T→MagneticFieldStrength=3, rest→TaskName=rest); "
    "include scanner synonyms (Siemens→TrioTim Prisma Skyra); "
    "use AND/OR/NOT boolean operators. "
    "Output ONLY a single line of space-separated search terms — no explanation, "
    "no JSON, no markdown.\n"
    "Query: {query}\nExpanded:"
)

SYNONYM_MAP = {
    "resting state":   "resting state rest TaskName=rest task=rest rsfMRI default-mode-network spontaneous",
    "resting-state":   "resting state rest TaskName=rest task=rest rsfMRI",
    "resting":         "resting rest TaskName=rest rsfMRI",
    "fmri":            "fMRI BOLD bold functional Blood-Oxygen-Level-Dependent suffix=bold",
    "functional":      "functional fMRI BOLD bold suffix=bold activation",
    "diffusion":       "diffusion DWI dwi DTI dti white-matter tractography b-value bvec bval",
    "dti":             "DTI DWI diffusion tensor imaging white matter b0",
    "dwi":             "DWI DTI diffusion weighted imaging b-value bvec bval",
    "anatomical":      "anatomical structural T1w T2w MPRAGE MP2RAGE brain morphology",
    "structural":      "structural anatomical T1w MPRAGE MP2RAGE suffix=T1w",
    "t1":              "T1w T1 structural anatomical MPRAGE UNIT1",
    "t2":              "T2w T2 structural spin-echo suffix=T2w",
    "flair":           "FLAIR T2 fluid attenuated inversion recovery",
    "mapping":         "quantitative qMRI relaxometry T1map T2map suffix=T1map InversionTime",
    "quantitative":    "quantitative qMRI relaxometry suffix=T1map InversionTime FlipAngle",
    "mp2rage":         "MP2RAGE UNIT1 suffix=UNIT1 inversion-recovery two-inversion ultra-high-field",
    "asl":             "ASL arterial-spin-labeling suffix=asl m0scan perfusion cerebral-blood-flow CBF",
    "perfusion":       "perfusion ASL arterial-spin-labeling suffix=asl m0scan CBF",
    "fieldmap":        "fieldmap phasediff magnitude1 magnitude2 epi distortion-correction B0 suffix=phasediff",
    "3t":              "3T 3Tesla MagneticFieldStrength=3 three-Tesla",
    "1.5t":            "1.5T MagneticFieldStrength=1.5",
    "7t":              "7T 7Tesla MagneticFieldStrength=7 ultra-high-field",
    "siemens":         "Siemens TrioTim Prisma Skyra Verio Manufacturer=Siemens",
    "philips":         "Philips Achieva Ingenia Manufacturer=Philips",
    "ge":              "GE Discovery Signa Manufacturer=GE General-Electric",
    "bold":            "BOLD fMRI bold suffix=bold functional Blood-Oxygen-Level-Dependent",
    "brain":           "brain neuroimaging MRI neural cortex cerebral",
    "high resolution": "high-resolution MPRAGE MP2RAGE 1mm isotropic sub-millimetre",
}


def _check_ollama() -> bool:
    try:
        _ureq.urlopen(_OLLAMA_HOST + "/api/tags", timeout=2)
        return True
    except Exception:
        return False


def _llm_expand(query: str, ollama_model: str = "llama3") -> str | None:
    try:
        payload = {"model": ollama_model,
                   "prompt": OLLAMA_PROMPT.format(query=query),
                   "stream": False}
        req = _ureq.Request(OLLAMA_URL, data=_json.dumps(payload).encode(),
                            headers={"Content-Type": "application/json"}, method="POST")
        with _ureq.urlopen(req, timeout=30) as resp:
            return _json.loads(resp.read()).get("response", "").strip()
    except Exception:
        return None


def _rule_expand(query: str) -> str:
    lower = query.lower()
    extra = [exp for kw, exp in SYNONYM_MAP.items()
             if re.search(r'\b' + re.escape(kw) + r'\b', lower)]
    return (query + " " + " ".join(extra)).strip() if extra else query


def expand_query(query: str) -> tuple[str, str]:
    """Return (expanded_text, backend_used). Tries Ollama first, falls back to rules."""
    llm = _llm_expand(query)
    if llm:
        return llm, "ollama"
    return _rule_expand(query), "rule-based"


# Status check
if _check_ollama():
    print(f"\u2705 Ollama is running at {_OLLAMA_HOST}")
    try:
        with _ureq.urlopen(_OLLAMA_HOST + "/api/tags", timeout=2) as resp:
            tags = _json.loads(resp.read())
        models = [m["name"] for m in tags.get("models", [])]
        if models:
            print(f"   Available models: {models}")
            if not any("llama3" in m for m in models):
                print("   \u26a0\ufe0f  llama3 not found — run: ollama pull llama3")
        else:
            print("   \u26a0\ufe0f  No models pulled yet — run: ollama pull llama3")
    except Exception:
        pass
else:
    print(
        f"\u26a0\ufe0f  Ollama not reachable at {_OLLAMA_HOST} — will use rule-based fallback")
    print("   To enable LLM expansion: docker exec -it ollama-dev ollama pull llama3")

# Smoke test — three neuroimaging query types
for q in [
    "resting state fMRI Siemens 3T",
    "quantitative T1 mapping MP2RAGE inversion recovery",
    "ASL arterial spin labeling perfusion 7T",
]:
    exp, backend = expand_query(q)
    print(f"\n[{backend}] {q!r}")
    print(f"  → {exp}")

✅ Ollama is running at http://ollama:11434
   Available models: ['llama3:latest']

[rule-based] 'resting state fMRI Siemens 3T'
  → resting state fMRI Siemens 3T resting state rest TaskName=rest task=rest rsfMRI default-mode-network spontaneous resting rest TaskName=rest rsfMRI fMRI BOLD bold functional Blood-Oxygen-Level-Dependent suffix=bold 3T 3Tesla MagneticFieldStrength=3 three-Tesla Siemens TrioTim Prisma Skyra Verio Manufacturer=Siemens

[rule-based] 'quantitative T1 mapping MP2RAGE inversion recovery'
  → quantitative T1 mapping MP2RAGE inversion recovery T1w T1 structural anatomical MPRAGE UNIT1 quantitative qMRI relaxometry T1map T2map suffix=T1map InversionTime quantitative qMRI relaxometry suffix=T1map InversionTime FlipAngle MP2RAGE UNIT1 suffix=UNIT1 inversion-recovery two-inversion ultra-high-field

[ollama] 'ASL arterial spin labeling perfusion 7T'
  → **asl AND MagneticFieldStrength=3 AND Manufacturer=(TrioTim OR Prisma OR Skyra)**


## Option D — Client-side Reciprocal Rank Fusion

ES 9.3 native RRF requires a paid license. We implement the identical algorithm
in ~10 lines of Python: **free, transparent, and portable**.

```
rrf_score(d) = Σ  1 / (k + rank_i(d))      k = 60  (smoothing constant)
```

A document ranked 1st contributes 1/(60+1) ≈ 0.016; ranked 10th contributes
1/(60+10) ≈ 0.014. The gap is small — which makes RRF robust to score scale
differences between BM25 and kNN.

In [4]:
def rrf_fuse(hit_lists: list[list[dict]], k: int = RRF_K) -> list[dict]:
    """
    Fuse multiple ES hit lists with Reciprocal Rank Fusion.

    Parameters
    ----------
    hit_lists : each element is response["hits"]["hits"]
    k         : smoothing constant (default 60)

    Returns a merged list of fake hit dicts sorted by RRF score.
    """
    scores: dict[str, float] = {}
    sources: dict[str, dict] = {}
    for hits in hit_lists:
        for rank, hit in enumerate(hits, start=1):
            doc_id = hit["_id"]
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            sources[doc_id] = hit.get("_source", {})
    return [
        {"_id": doc_id, "_score": score, "_source": sources[doc_id]}
        for doc_id, score in sorted(scores.items(), key=lambda x: -x[1])
    ]


def weighted_fuse(bm25_hits, knn_hits, bm25_weight=0.3, knn_weight=0.7) -> list[dict]:
    """Simple weighted score fusion — used when USE_RRF=False."""
    # Normalise each list to [0, 1] then combine
    def normalise(hits):
        scores = [h["_score"] or 0 for h in hits]
        mx = max(scores) if scores else 1.0
        return {h["_id"]: (h["_score"] or 0) / (mx or 1.0) for h in hits}

    b = normalise(bm25_hits)
    v = normalise(knn_hits)
    all_ids = set(b) | set(v)
    sources = {h["_id"]: h.get("_source", {})
               for hits in (bm25_hits, knn_hits) for h in hits}
    fused = {doc_id: bm25_weight * b.get(doc_id, 0) + knn_weight * v.get(doc_id, 0)
             for doc_id in all_ids}
    return [
        {"_id": doc_id, "_score": score, "_source": sources[doc_id]}
        for doc_id, score in sorted(fused.items(), key=lambda x: -x[1])
    ]

## Index Setup — `neuroimaging-poc1`

We read every document from the base `neuroimaging` index, re-embed
`description_text` with the better encoder, and write to a new index.

> **One-time cost:** ~5 min on CPU for 3 000 documents.  
> The cell is idempotent — re-running skips creation if the index already exists.

In [5]:
POC1_MAPPINGS = {
    "properties": {
        "dataset":   {"type": "keyword"},
        "subject":   {"type": "keyword"},
        "session":   {"type": "keyword"},
        "task":      {"type": "keyword"},
        "run":       {"type": "keyword"},
        "suffix":    {"type": "keyword"},
        "datatype":  {"type": "keyword"},
        "age":       {"type": "float"},
        "sex":       {"type": "keyword"},
        "RepetitionTime":        {"type": "float"},
        "EchoTime":              {"type": "float"},
        "FlipAngle":             {"type": "float"},
        "MagneticFieldStrength": {"type": "float"},
        "SliceThickness":        {"type": "float"},
        "InversionTime":         {"type": "float"},
        "Manufacturer":           {"type": "keyword"},
        "ManufacturersModelName": {"type": "keyword"},
        "InstitutionName":        {"type": "keyword"},
        "PhaseEncodingDirection": {"type": "keyword"},
        "TaskName":          {"type": "text"},
        "SeriesDescription": {"type": "text"},
        "description_text":  {"type": "text"},
        "metadata_embedding": {
            "type": "dense_vector",
            "dims": EMBEDDING_DIMS,
            "similarity": "cosine",
            "index_options": {"type": "int8_hnsw"},
        },
        "bids_path": {"type": "keyword"},
    }
}

if client.indices.exists(index=POC1_INDEX):
    doc_count = client.count(index=POC1_INDEX)["count"]
    print(
        f"Index {POC1_INDEX!r} already exists with {doc_count} docs — skipping creation.")
    print("Delete and re-run this cell to force re-index:")
    print(f"  client.indices.delete(index={POC1_INDEX!r})")
else:
    # Verify source index exists
    if not client.indices.exists(index=BASE_INDEX):
        raise RuntimeError(
            f"Base index {BASE_INDEX!r} not found. Run Notebook 01 first "
            f"(pointing it to {ES_HOST})."
        )
    client.indices.create(
        index=POC1_INDEX,
        settings={"number_of_replicas": 0},
        mappings=POC1_MAPPINGS
    )
    print(
        f"Created index {POC1_INDEX!r} with {EMBEDDING_DIMS}-dim dense_vector field.")

Index 'neuroimaging-poc1' already exists with 4423 docs — skipping creation.
Delete and re-run this cell to force re-index:
  client.indices.delete(index='neuroimaging-poc1')


In [6]:
# ── Ingest: fetch → re-embed → bulk-index ────────────────────────────────────

def fetch_all_docs(index: str, batch_size: int = 500) -> list[dict]:
    """Paginate through index, excluding heavy embedding field."""
    docs, from_ = [], 0
    while True:
        resp = client.search(
            index=index, query={"match_all": {}},
            size=batch_size, from_=from_,
            source_excludes=["metadata_embedding", "sparse_embedding"]
        )
        hits = resp["hits"]["hits"]
        if not hits:
            break
        docs.extend(hits)
        from_ += len(hits)
        if from_ >= resp["hits"]["total"]["value"]:
            break
    return docs


def batches(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


if client.count(index=POC1_INDEX)["count"] == 0:
    print(f"Fetching documents from {BASE_INDEX!r} ...")
    base_docs = fetch_all_docs(BASE_INDEX)
    print(
        f"Fetched {len(base_docs)} documents. Embedding with {ENCODER_MODEL} ...")

    actions = []
    for batch in tqdm(list(batches(base_docs, 32)), desc="Embedding batches"):
        texts = [d["_source"].get("description_text", "") for d in batch]
        vectors = model.encode(texts, show_progress_bar=False).tolist()
        for doc, vec in zip(batch, vectors):
            src = doc["_source"].copy()
            src["metadata_embedding"] = vec
            actions.append(
                {"_index": POC1_INDEX, "_id": doc["_id"], "_source": src})

    helpers.bulk(client, actions, chunk_size=200)
    client.indices.refresh(index=POC1_INDEX)
    print(f"Indexed {len(actions)} docs into {POC1_INDEX!r}")
else:
    print(f"Index {POC1_INDEX!r} already populated — skipping ingest.")

print(
    f"Total docs in {POC1_INDEX!r}: {client.count(index=POC1_INDEX)['count']}")

Index 'neuroimaging-poc1' already populated — skipping ingest.
Total docs in 'neuroimaging-poc1': 4423


## Search Functions

Three search modes, all using `expand_query()` on the input:

| Mode | Query channels | Fusion |
|------|---------------|--------|
| `bm25` | BM25 on `description_text` | — |
| `knn` | dense kNN on `metadata_embedding` | — |
| `hybrid` | BM25 + kNN | RRF (if `USE_RRF`) or weighted |

Pass `expand=False` to bypass query expansion (for ablation comparisons).

In [7]:
def embed(text: str) -> list[float]:
    return model.encode(text, show_progress_bar=False).tolist()


def poc1_search(
    query: str,
    mode: str = "hybrid",   # "bm25" | "knn" | "hybrid"
    expand: bool = True,
    k: int = TOP_K,
    use_rrf: bool = USE_RRF,
    num_candidates: int = 100,
) -> tuple[list[dict], dict]:
    """
    Run PoC-1 search.

    Returns
    -------
    hits   : list of hit dicts with _id, _score, _source
    meta   : {"query", "expanded", "backend", "mode", "latency_ms"}
    """
    t0 = time.perf_counter()
    expanded, backend = expand_query(query) if expand else (query, "none")

    if mode == "bm25":
        resp = client.search(
            index=POC1_INDEX,
            query={"match": {"description_text": expanded}},
            size=k
        )
        hits = resp["hits"]["hits"]

    elif mode == "knn":
        resp = client.search(
            index=POC1_INDEX,
            knn={"field": "metadata_embedding",
                 "query_vector": embed(expanded),
                 "k": k, "num_candidates": num_candidates},
        )
        hits = resp["hits"]["hits"]

    else:  # hybrid
        if use_rrf:
            vec = embed(expanded)
            bm25_resp = client.search(
                index=POC1_INDEX,
                query={"match": {"description_text": expanded}},
                size=k * 3
            )
            knn_resp = client.search(
                index=POC1_INDEX,
                knn={"field": "metadata_embedding", "query_vector": vec,
                     "k": k * 3, "num_candidates": num_candidates * 2},
            )
            hits = rrf_fuse(
                [bm25_resp["hits"]["hits"], knn_resp["hits"]["hits"]]
            )[:k]
        else:
            vec = embed(expanded)
            bm25_resp = client.search(
                index=POC1_INDEX,
                query={"match": {"description_text": expanded}},
                size=k * 3
            )
            knn_resp = client.search(
                index=POC1_INDEX,
                knn={"field": "metadata_embedding", "query_vector": vec,
                     "k": k * 3, "num_candidates": num_candidates * 2},
            )
            hits = weighted_fuse(
                bm25_resp["hits"]["hits"], knn_resp["hits"]["hits"]
            )[:k]

    latency_ms = (time.perf_counter() - t0) * 1000
    meta = {"query": query, "expanded": expanded, "backend": backend,
            "mode": mode, "latency_ms": round(latency_ms, 1)}
    return hits, meta

## Demo — Ablation: No Expansion vs. Expansion vs. Full PoC-1

We run the same query through four configurations to isolate each contribution.

In [ ]:
# ── Context-aware column presets ────────────────────────────────────────────
COLS_FUNCTIONAL = ["dataset", "suffix", "subject", "task", "TaskName",
                   "RepetitionTime", "EchoTime", "MagneticFieldStrength",
                   "Manufacturer"]
COLS_STRUCTURAL = ["dataset", "suffix", "subject", "MagneticFieldStrength",
                   "Manufacturer", "ManufacturersModelName", "InversionTime",
                   "FlipAngle", "MRAcquisitionType"]
COLS_DIFFUSION = ["dataset", "suffix", "subject", "MagneticFieldStrength",
                  "Manufacturer", "PhaseEncodingDirection", "SliceThickness"]
COLS_SCANNER = ["dataset", "suffix", "subject", "Manufacturer",
                "ManufacturersModelName", "MagneticFieldStrength",
                "InstitutionName"]

# ── Demo queries: grounded in the actual data in this corpus ─────────────────
# Each query is paired with the column preset that best surfaces the relevant
# metadata for a student to understand WHY a result matches.
DEMO_QUERIES = [
    # (query_text, column_preset, description)
    ("resting state fMRI on a Siemens 3T scanner",
     COLS_FUNCTIONAL,
     "Classic rs-fMRI search: should rank ds000117, 7t_trt, ds001 BOLD scans"),

    ("diffusion weighted imaging white matter tractography microstructure",
     COLS_DIFFUSION,
     "DWI search: should return suffix=dwi across ds002, ds003, ds114 etc."),

    ("quantitative T1 mapping inversion recovery MPRAGE MP2RAGE",
     COLS_STRUCTURAL,
     "qMRI: should surface qmri_mp2rage, qmri_irt1, qmri_mese datasets"),

    ("arterial spin labeling cerebral blood flow perfusion",
     COLS_FUNCTIONAL,
     "ASL: should return asl001–asl005, 2d_mb_pcasl datasets"),

    ("7 Tesla ultra-high field brain imaging",
     COLS_SCANNER,
     "7T search: should surface 7t_trt and qMRI datasets with B1 mapping"),
]


def demo_all(queries, poc1_search_fn):
    """Run all demo queries and print structured comparison."""
    for q, cols, description in queries:
        print(f"\n{'='*72}")
        print(f"Query:  {q!r}")
        print(f"Expect: {description}")

        configs = [
            ("BM25 (no expand)",         "bm25",   False, False),
            ("BM25 (+ expand)",           "bm25",   True,  False),
            ("kNN  (+ expand)",           "knn",    True,  False),
            ("Hybrid + RRF (full PoC-1)", "hybrid", True,  True),
        ]

        for label, mode, expand_, use_rrf_ in configs:
            hits, meta = poc1_search_fn(q, mode=mode, expand=expand_,
                                        use_rrf=use_rrf_, k=10)
            scores = [round(float(h["_score"] or 0), 4) for h in hits]
            suffixes_seen = list(dict.fromkeys(
                h["_source"].get("suffix", "?") for h in hits))[:4]
            datasets_seen = list(dict.fromkeys(
                h["_source"].get("dataset", "?") for h in hits))[:4]
            print(f"  [{label:40s}] {meta['latency_ms']:6.1f} ms")
            print(f"    scores  : {scores}")
            print(f"    suffix  : {suffixes_seen}")
            print(f"    datasets: {datasets_seen}")
            if expand_ and meta["expanded"] != q:
                print(f"    expanded: {meta['expanded'][:80]}…")

        # Display full results for the best configuration (RRF)
        hits_rrf, _ = poc1_search_fn(q, mode="hybrid", expand=True,
                                     use_rrf=True, k=10)
        df = show_hits(hits_rrf, cols)
        print(f"\n  ── Full results (Hybrid+RRF, top-10) ──")
        display(df)


# ── Also add a multi-field BM25 variant for comparison ──────────────────────
def poc1_search_multifield(
    query: str,
    expand: bool = True,
    k: int = TOP_K,
    num_candidates: int = 100,
) -> tuple[list[dict], dict]:
    """
    Hybrid search where the BM25 leg queries MULTIPLE text fields with boosting.

    Motivation: the description_text field combines all metadata into one blob;
    boosting individual high-signal fields (TaskName, SeriesDescription)
    directly improves precision for queries that target those fields.
    """
    import time
    t0 = time.perf_counter()
    expanded, backend = expand_query(query) if expand else (query, "none")

    # Multi-field BM25: description_text × 1 + TaskName × 2 + SeriesDescription × 1.5
    bm25_resp = client.search(
        index=POC1_INDEX,
        query={
            "multi_match": {
                "query":  expanded,
                "fields": ["description_text", "TaskName^2",
                           "SeriesDescription^1.5", "TaskDescription"],
                "type":   "cross_fields",
                "operator": "or"
            }
        },
        size=k * 3
    )
    knn_resp = client.search(
        index=POC1_INDEX,
        knn={"field": "metadata_embedding",
             "query_vector": embed(expanded),
             "k": k * 3, "num_candidates": num_candidates * 2},
    )
    hits = rrf_fuse([bm25_resp["hits"]["hits"],
                     knn_resp["hits"]["hits"]])[:k]
    latency_ms = (time.perf_counter() - t0) * 1000
    meta = {"query": query, "expanded": expanded, "backend": backend,
            "mode": "hybrid_multifield", "latency_ms": round(latency_ms, 1)}
    return hits, meta


print("Demo queries and multi-field search functions ready.")
print(f"Corpus: {POC1_INDEX!r}")

Demo queries and multi-field search functions ready.
Corpus: 'neuroimaging-poc1'


## Save Results for Benchmark

Run all benchmark queries through PoC-1 and persist the raw hits to disk.
Notebook 07 will load these to compare against PoC-2.

In [ ]:
# ── Gold queries: ground-truth evaluation set (shared with Notebook 07) ─────
# These queries are intentionally HARDER than single-field lookups:
# - Multi-attribute (scanner + modality + protocol)
# - Clinical intent (what a researcher would actually type)
# - Quantitative threshold checks (e.g. "is TR in the fMRI range?")
# The check function defines what a "relevant" document looks like.

GOLD_QUERIES = [
    # ── Modality retrieval ─────────────────────────────────────────────────
    {
        "query":  "Blood Oxygen Level Dependent fMRI task activation brain",
        "label":  "BOLD / fMRI",
        "check": lambda s: s.get("suffix") == "bold",
        "note":   "Any BOLD scan — tests broad functional retrieval"
    },
    {
        "query":  "T1-weighted MPRAGE structural anatomical brain scan",
        "label":  "T1w structural",
        "check": lambda s: s.get("suffix") in ("T1w", "UNIT1", "FLASH"),
        "note":   "Structural MRI — includes MP2RAGE-derived UNIT1"
    },
    {
        "query":  "diffusion tensor DWI white-matter tractography b-value bvec",
        "label":  "DWI / diffusion",
        "check": lambda s: s.get("suffix") == "dwi",
        "note":   "Specifically tests DWI retrieval (suffix=dwi)"
    },
    {
        "query":  "phase-difference fieldmap B0 distortion correction EPI\'s",
        "label":  "Fieldmap (phasediff/epi)",
        "check": lambda s: s.get("suffix") in ("phasediff", "epi",
                                               "magnitude1", "magnitude2",
                                               "fieldmap"),
        "note":   "Fieldmap retrieval — tests multi-suffix grouping"
    },
    # ── Scanner metadata ───────────────────────────────────────────────────
    {
        "query":  "Siemens MRI scanner TrioTim Prisma Skyra neuroimaging",
        "label":  "Siemens scanner",
        "check": lambda s: "siemens" in str(s.get("Manufacturer", "")).lower(),
        "note":   "Tests manufacturer retrieval, includes model names"
    },
    {
        "query":  "3 Tesla high-field MRI neuroimaging research scanner",
        "label":  "3T field strength",
        "check": lambda s: s.get("MagneticFieldStrength") == 3.0,
        "note":   "Field strength: should NOT return 1.5T or 7T documents"
    },
    {
        "query":  "ultra-high field 7 Tesla MRI B1 transmit homogeneity",
        "label":  "7T field strength",
        "check": lambda s: s.get("MagneticFieldStrength") == 7.0,
        "note":   "Specifically targets 7T — useful for quantitative MRI"
    },
    # ── Task / paradigm ─────────────────────────────────────────────────
    {
        "query":  "resting state eyes-open spontaneous brain activity fMRI rest",
        "label":  "Resting-state fMRI",
        "check": lambda s: (
            s.get("suffix") == "bold"
            and ("rest" in str(s.get("task", "")).lower()
                 or "rest" in str(s.get("TaskName", "")).lower())
        ),
        "note":   "Task+suffix combo: must be bold AND rest"
    },
    {
        "query":  "cognitive task fMRI activation experimental paradigm",
        "label":  "Task-fMRI (non-rest)",
        "check": lambda s: (
            s.get("suffix") == "bold"
            and s.get("task") not in (None, "", "rest", "restingstate",
                                      "restingstate01", "rest1")
        ),
        "note":   "Task fMRI: BOLD scan tied to an explicit non-rest task"
    },
    # ── Quantitative / advanced ────────────────────────────────────────
    {
        "query":  "quantitative MRI relaxometry T1 mapping inversion recovery MP2RAGE",
        "label":  "qMRI (T1map/UNIT1)",
        "check": lambda s: s.get("suffix") in ("T1map", "UNIT1", "IRT1",
                                               "VFA", "T1w")
        and s.get("InversionTime") is not None,
        "note":   "qMRI: needs InversionTime field to be present"
    },
    {
        "query":  "arterial spin labeling perfusion cerebral blood flow ASL",
        "label":  "ASL / perfusion",
        "check": lambda s: s.get("suffix") in ("asl", "m0scan"),
        "note":   "Tests rare modality retrieval (ASL datasets)"
    },
    {
        "query":  "short repetition time multiband EPI fast fMRI acquisition",
        "label":  "Short TR BOLD (TR < 1.5s)",
        "check": lambda s: (
            s.get("suffix") == "bold"
            and s.get("RepetitionTime") is not None
            and float(s.get("RepetitionTime") or 99) < 1.5
        ),
        "note":   "Tests TR range + modality combo — multiband datasets"
    },
]


def precision_at_k(hits, check_fn, k):
    top = hits[:k]
    return sum(1 for h in top if check_fn(h["_source"])) / k if top else 0.0


def mean_reciprocal_rank(hits, check_fn, max_rank=10):
    for rank, hit in enumerate(hits[:max_rank], start=1):
        if check_fn(hit["_source"]):
            return 1.0 / rank
    return 0.0


def score_spread(hits):
    scores = [float(h.get("_score") or 0) for h in hits]
    return float(np.std(scores)) if len(scores) > 1 else 0.0


print(f"Evaluation set ready: {len(GOLD_QUERIES)} queries")
for gq in GOLD_QUERIES:
    print(f"  [{gq['label']:30s}] {gq['note']}")

Evaluation set ready: 12 queries
  [BOLD / fMRI                   ] Any BOLD scan — tests broad functional retrieval
  [T1w structural                ] Structural MRI — includes MP2RAGE-derived UNIT1
  [DWI / diffusion               ] Specifically tests DWI retrieval (suffix=dwi)
  [Fieldmap (phasediff/epi)      ] Fieldmap retrieval — tests multi-suffix grouping
  [Siemens scanner               ] Tests manufacturer retrieval, includes model names
  [3T field strength             ] Field strength: should NOT return 1.5T or 7T documents
  [7T field strength             ] Specifically targets 7T — useful for quantitative MRI
  [Resting-state fMRI            ] Task+suffix combo: must be bold AND rest
  [Task-fMRI (non-rest)          ] Task fMRI: BOLD scan tied to an explicit non-rest task
  [qMRI (T1map/UNIT1)            ] qMRI: needs InversionTime field to be present
  [ASL / perfusion               ] Tests rare modality retrieval (ASL datasets)
  [Short TR BOLD (TR < 1.5s)     ] Tests TR

## Summary — PoC-1 Architecture

```
User query
   │
   ▼
expand_query()  [B]             rule-based synonym map or Ollama LLM
   │
   ├──► BM25 match  ──────────────────────► top-3k hits
   │    (description_text)                      │
   │                                            │
   └──► all-mpnet-base-v2 [A]                   │
           │                                    │
           ▼                                    │
        kNN search ──────────────────────► top-3k hits
        (metadata_embedding 768d)               │
                                                │
                                          rrf_fuse() [D]
                                                │
                                          top-k results
```

**Next step:** Open Notebook 07 to see PoC-2 (SPLADE sparse + RRF) and the
full head-to-head benchmark.